# Weekend vs Weekday classification based on energy usage with XGBoost

In [1]:
from pathlib import Path
import os
import time


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy as sp_entropy
from xgboost import XGBClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay,
)


## Data Loading

In [2]:
DATA_PATH = Path("dataset/LD2011_2014.txt")
OUTPUT_DIR = Path("outputs/classification_all")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df= pd.read_csv(
    DATA_PATH,
    delimiter=";",
    decimal=",",
    parse_dates=[0],
    index_col=0,
    low_memory=False,
)

df.index.name = "Time"
df = df.sort_index() 

print(f"Loaded:  {df.shape[0]:,} rows  ×  {df.shape[1]} clients")
print(f"Range:   {df.index.min()}  →  {df.index.max()}")
df.head()




Loaded:  140,256 rows  ×  370 clients
Range:   2011-01-01 00:15:00  →  2015-01-01 00:00:00


,MT_001,MT_002,MT_003,MT_004,MT_005,MT_006,MT_007,MT_008,MT_009,MT_010,...,MT_361,MT_362,MT_363,MT_364,MT_365,MT_366,MT_367,MT_368,MT_369,MT_370
Time,,,,,,,,,,,,,,,,,,,,,
2011-01-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 00:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2011-01-01 01:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
dates = df.index.normalize().unique()
slots_per_day = df.groupby(df.index.normalize()).size()
 
unexpected = slots_per_day[slots_per_day != 96]
if unexpected.empty:
    print("\nDays with exactly 96 slots")
else:
    print(f"\nDays with less/more than 96 slots ({len(unexpected)}):")
    print(unexpected)


Days with less/more than 96 slots (2):
Time
2011-01-01    95
2015-01-01     1
dtype: int64


## Preprocessing
Preprocessing decisions:
- drop partial days ("2011-01-01", "2015-01-01")
- replace client data beofre their activation with NaN
- unpivot data for xgboost 
- introduce Weekday vs Weekend label for classification

In [4]:
DROP_DAYS = ["2011-01-01", "2015-01-01"]
df = df[~df.index.normalize().astype(str).isin(DROP_DAYS)]
df = df.asfreq("15min")  
 
print(f"After boundary drop + asfreq: {df.shape}")
print(f"Index dtype: {df.index.dtype}  |  tz: {df.index.tz}")



After boundary drop + asfreq: (140160, 370)
Index dtype: datetime64[us]  |  tz: None


In [5]:
first_active = {}
for col in df.columns:
    nonzero_idx = df.index[df[col] > 0]
    first_active[col] = nonzero_idx[0] if len(nonzero_idx) > 0 else pd.NaT
 
first_active_s = pd.Series(first_active, name="first_active")
dataset_start  = df.index.min()
 
print(f"On-time clients : {(first_active_s == dataset_start).sum()}")
print(f"Late starters   : {(first_active_s > dataset_start).sum()}")

hourly = df.resample("h").mean()
hourly.index.name = "Time"
print(f"\nHourly shape (before masking): {hourly.shape}")


hourly_masked = hourly.copy()
 
for col, fa in first_active_s.items():
    if pd.isna(fa):
        hourly_masked[col] = np.nan
        continue
    fa_hour = fa.floor("h")
    hourly_masked.loc[hourly_masked.index < fa_hour, col] = np.nan
 
print(f"NaN cells after masking: {hourly_masked.isna().sum().sum():,}")



On-time clients : 158
Late starters   : 212

Hourly shape (before masking): (35040, 370)
NaN cells after masking: 2,484,474


In [6]:
clients   = hourly_masked.columns.tolist()
n_clients = len(clients)
 
dates  = hourly_masked.index.normalize().unique().sort_values()
n_days = len(dates)
 
assert len(hourly_masked) == n_days * 24, (
    f"Hourly rows ({len(hourly_masked)}) != n_days×24 ({n_days*24}). "
    "Check for missing hours after asfreq."
)
 

arr = hourly_masked.values.reshape(n_days, 24, n_clients)
arr = arr.transpose(2, 0, 1)
arr_2d = arr.reshape(n_clients * n_days, 24)
 

client_idx = np.repeat(clients, n_days)
date_idx   = np.tile(dates, n_clients)
 
hour_cols = [f"h{h:02d}" for h in range(24)]
wide = pd.DataFrame(arr_2d, columns=hour_cols)
wide.insert(0, "date",   date_idx)
wide.insert(0, "client", client_idx)
 
print(f"\nWide shape: {wide.shape}")



Wide shape: (540200, 26)


In [7]:
wide["date"]  = pd.to_datetime(wide["date"])
wide["label"] = (wide["date"].dt.dayofweek >= 5).astype(int)
 
print(f"\nLabel distribution:")
vc = wide["label"].value_counts().rename({0: "Weekday", 1: "Weekend"})
print(vc)
print(f"Ratio weekday/weekend: {vc['Weekday']/vc['Weekend']:.2f}")



Label distribution:
label
Weekday    385910
Weekend    154290
Name: count, dtype: int64
Ratio weekday/weekend: 2.50


In [8]:
wide["date"]  = pd.to_datetime(wide["date"])
wide["label"] = (wide["date"].dt.dayofweek >= 5).astype(int)
 
print(f"\nLabel distribution:")
vc = wide["label"].value_counts().rename({0: "Weekday", 1: "Weekend"})
print(vc)
print(f"Ratio weekday/weekend: {vc['Weekday']/vc['Weekend']:.2f}")



Label distribution:
label
Weekday    385910
Weekend    154290
Name: count, dtype: int64
Ratio weekday/weekend: 2.50


In [9]:
wide.tail(5)

,client,date,h00,h01,h02,h03,h04,h05,h06,h07,...,h15,h16,h17,h18,h19,h20,h21,h22,h23,label
540195,MT_370,2014-12-27,14891.891892,14054.054054,12662.162162,13121.621622,14175.675676,12891.891892,16378.378378,15189.189189,...,15662.162162,16040.540541,15256.756757,15432.432432,15959.459459,15486.486486,15770.270270,14283.783784,13162.162162,1
540196,MT_370,2014-12-28,14378.378378,14135.135135,14472.972973,15256.756757,15770.270270,14445.945946,15405.405405,15810.810811,...,18797.297297,17513.513514,19500.000000,17932.432432,15986.486486,15959.459459,17513.513514,17243.243243,17121.621622,1
540197,MT_370,2014-12-29,17716.216216,16932.432432,17986.486486,16635.135135,18256.756757,18918.918919,16648.648649,19770.270270,...,17824.324324,21189.189189,21864.864865,21283.783784,20054.054054,20054.054054,17081.081081,19283.783784,18918.918919,0
540198,MT_370,2014-12-30,17824.324324,19743.243243,17837.837838,17945.945946,18648.648649,18337.837838,18621.621622,19581.081081,...,20108.108108,21500.000000,20486.486486,19594.594595,20513.513514,19135.135135,19121.621622,18743.243243,18054.054054,0
540199,MT_370,2014-12-31,17351.351351,17972.972973,17337.837838,17216.216216,16581.081081,16243.243243,15932.432432,16675.675676,...,14378.378378,12459.459459,11608.108108,10135.135135,8972.972973,8405.405405,8283.783784,7594.594595,6932.432432,0


## Train/test split  
- Train : 2011-01-02 -> 2014-06-30  (no shuffle)
- Test  : 2014-07-01 -> 2014-12-31

## Feature engineering

The following features were engineered prior to modeling:

* **Daily Aggregates:** Computed basic descriptive statistics for each 24-hour cycle including `daily_total`, `daily_peak`, `daily_trough`, `daily_mean`, and `daily_std`.
* **Time-of-Day Windows:** Segmented the 24 hours into specific behavioral blocks to extract contextual consumption trends:
* *Night:* Hours 0–5 (`night_mean`)
* *Morning:* Hours 6–9 (`morning_mean`)
* *Midday:* Hours 10–14 (`midday_mean`)
* *Evening:* Hours 17–21 (`evening_mean`)


* **Consumption Ratios:** `morning_to_night_ratio`, `evening_to_midday_ratio`, `peak_to_mean_ratio`.
* **Peak Hour Tracking:** Identified the exact hour of maximum consumption (`peak_hour`) for each client day.
* **Hourly Entropy (`hourly_entropy`):** Quantified the randomness/flatness of hourly energy usage across the day using a normalized vector-based entropy calculation.
* **Historical Lag Features:** Captured short-term and weekly auto-regressive patterns by shifting client records chronologically:
* *Short-term:* 1-day (`lag1`) and 2-day (`lag2`) to compare usage to previous days.
* *Weekly seasonality:* A 7-day (`lag7`) to compare usage to the exact same day of the prior week.


* **Cyclical & Calendar Seasonality:** * Transformed the calendar month using Sine and Cosine functions (`month_sin`, `month_cos`).
* Generated dummy variables for seasons (*Winter, Spring, Summer, Autumn*).
* Flagged month start/end boundaries (`is_month_boundary`) to capture billing or calendar-specific resets.


* **Behavioral Segment Distance (`wd_we_l2`):** Calculated the Euclidean distance (L2 norm) between each client's average weekday and weekend profiles using the training subset. This metric categorizes clients into behavioral groups (*Flat, Moderate, Strong*) to enable segmented model training.

In [10]:
client_day_df = wide.copy()
SPLIT_DATE = pd.Timestamp("2014-07-01")
hour_cols  = [f"h{h:02d}" for h in range(24)]


df = client_day_df.copy()
df["date"] = pd.to_datetime(df["date"])


df = df.dropna(subset=hour_cols, how="all")
h  = df[hour_cols].values.astype(float)
eps = 1e-6

# Feature engineering
# Daily stats
df["daily_total"]  = np.nansum(h,  axis=1)
df["daily_peak"]   = np.nanmax(h,  axis=1)
df["daily_trough"] = np.nanmin(h,  axis=1)
df["daily_mean"]   = np.nanmean(h, axis=1)
df["daily_std"]    = np.nanstd(h,  axis=1)

# Time-of-day windows
df["night_mean"]   = np.nanmean(h[:, 0:6],   axis=1)
df["morning_mean"] = np.nanmean(h[:, 6:10],  axis=1)
df["midday_mean"]  = np.nanmean(h[:, 10:15], axis=1)
df["evening_mean"] = np.nanmean(h[:, 17:22], axis=1)

# Ratios
df["morning_to_night_ratio"]  = df["morning_mean"] / (df["night_mean"]  + eps)
df["evening_to_midday_ratio"] = df["evening_mean"] / (df["midday_mean"] + eps)
df["peak_to_mean_ratio"]      = df["daily_peak"]   / (df["daily_mean"]  + eps)

# Peak hour
df["peak_hour"] = np.argmax(
    np.where(np.isnan(h), -np.inf, h), axis=1).astype(float)
df.loc[df[hour_cols].isna().all(axis=1), "peak_hour"] = np.nan


def row_entropy(row):
    vals = row[row > 0]
    if len(vals) < 2:
        return 0.0
    p = vals / vals.sum()
    return sp_entropy(p)

df["hourly_entropy"] = (
    df[hour_cols]
    .apply(row_entropy, axis=1)
)

agg_cols = [
    "daily_total", "daily_peak", "daily_trough", "daily_mean", "daily_std",
    "night_mean", "morning_mean", "midday_mean", "evening_mean",
    "morning_to_night_ratio", "evening_to_midday_ratio",
    "peak_to_mean_ratio", "peak_hour", "hourly_entropy",
]



C:\Users\katar\AppData\Local\Temp\ipykernel_25776\982616316.py:23: RuntimeWarning: Mean of empty slice
  df["night_mean"]   = np.nanmean(h[:, 0:6],   axis=1)
C:\Users\katar\AppData\Local\Temp\ipykernel_25776\982616316.py:24: RuntimeWarning: Mean of empty slice
  df["morning_mean"] = np.nanmean(h[:, 6:10],  axis=1)
C:\Users\katar\AppData\Local\Temp\ipykernel_25776\982616316.py:25: RuntimeWarning: Mean of empty slice
  df["midday_mean"]  = np.nanmean(h[:, 10:15], axis=1)


### Lag features

In [11]:
df = df.sort_values(["client", "date"]).reset_index(drop=True)

lag_source = ["daily_total", "daily_mean", "daily_std",
              "morning_mean", "evening_mean", "peak_hour"]

for col in lag_source:
    df[f"lag1_{col}"] = df.groupby("client")[col].shift(1)
    df[f"lag2_{col}"] = df.groupby("client")[col].shift(2)

lag7_source = ["daily_total", "daily_mean", "daily_std"]
for col in lag7_source:
    df[f"lag7_{col}"] = df.groupby("client")[col].shift(7)

lag1_cols = [f"lag1_{c}" for c in lag_source]
lag2_cols = [f"lag2_{c}" for c in lag_source]
lag7_cols = [f"lag7_{c}" for c in lag7_source]

print(f"Lag NaN rows (expected — first days per client):")
print(f"  lag1: {df[lag1_cols].isna().any(axis=1).sum():,}")
print(f"  lag2: {df[lag2_cols].isna().any(axis=1).sum():,}")
print(f"  lag7: {df[lag7_cols].isna().any(axis=1).sum():,}")

Lag NaN rows (expected — first days per client):
  lag1: 386
  lag2: 756
  lag7: 2,590


### Seasonal features

In [12]:

month = df["date"].dt.month
df["month_sin"] = np.sin(2 * np.pi * month / 12)
df["month_cos"] = np.cos(2 * np.pi * month / 12)

season_map = {12:"winter",1:"winter",2:"winter",
              3:"spring",4:"spring",5:"spring",
              6:"summer",7:"summer",8:"summer",
              9:"autumn",10:"autumn",11:"autumn"}
season_dummies = pd.get_dummies(
    month.map(season_map), prefix="season", dtype=float)
df = pd.concat([df, season_dummies], axis=1)
season_cols = season_dummies.columns.tolist()

day = df["date"].dt.day
df["is_month_boundary"] = ((day <= 3) | (day >= df["date"].dt.days_in_month - 2)).astype(float)

seasonal_cols = ["month_sin", "month_cos"] + season_cols + ["is_month_boundary"]


In [13]:
base_feature_cols = hour_cols + agg_cols + lag1_cols + lag2_cols + lag7_cols + seasonal_cols
print(f"\nBase feature count: {len(base_feature_cols)}")


Base feature count: 60


## Train / test split

In [14]:
train_all = df[df["date"] <  SPLIT_DATE].copy()
test_all  = df[df["date"] >= SPLIT_DATE].copy()

print(f"Train: {len(train_all):,}  Test: {len(test_all):,}")

Train: 368,626  Test: 68,063


## Weekday vs. Weekend L2 Distance

In [15]:
def compute_l2(train_df, clients):
    rows = []
    for c in clients:
        sub = train_df[train_df["client"] == c][hour_cols + ["label"]].dropna(subset=hour_cols)
        if len(sub) < 20:
            rows.append({"client": c, "wd_we_l2": 0.0, "n_days": len(sub)})
            continue
        wd = sub[sub["label"] == 0][hour_cols].mean().values
        we = sub[sub["label"] == 1][hour_cols].mean().values
        rows.append({"client": c, "wd_we_l2": round(np.sqrt(np.nansum((wd - we) ** 2)), 3),
                     "n_days": len(sub)})
    return pd.DataFrame(rows).set_index("client")

all_clients  = df["client"].unique()
client_stats = compute_l2(train_all, all_clients)

t33 = client_stats["wd_we_l2"].quantile(0.33)
t67 = client_stats["wd_we_l2"].quantile(0.67)

def assign_segment(l2):
    if l2 <= t33: return "flat"
    if l2 <= t67: return "moderate"
    return "strong"

client_stats["segment"] = client_stats["wd_we_l2"].apply(assign_segment)

print(f"\nSegmentation thresholds: flat ≤ {t33:.2f}  moderate ≤ {t67:.2f}  strong > {t67:.2f}")
print(client_stats["segment"].value_counts().to_string())



Segmentation thresholds: flat ≤ 23.05  moderate ≤ 87.06  strong > 87.06
segment
moderate    126
flat        122
strong      122


In [16]:
def evaluate(model, X, y, label=""):
    prob = model.predict_proba(X)[:, 1]
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.30, 0.80, 0.01):
        f1 = f1_score(y, (prob >= t).astype(int), average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, round(t, 2)
    y_pred = (prob >= best_t).astype(int)
    results = {
        "best_t"    : best_t,
        "macro_f1"  : round(best_f1, 4),
        "weekend_f1": round(f1_score(y, y_pred, pos_label=1, zero_division=0), 4),
        "roc_auc"   : round(roc_auc_score(y, prob), 4),
        "accuracy"  : round((y_pred == y).mean(), 4),
        "y_pred"    : y_pred,
        "prob"      : prob,
    }
    if label:
        print(f"\n  {label}")
        print(f"  t={best_t}  macro-F1={results['macro_f1']}  "
              f"weekend-F1={results['weekend_f1']}  "
              f"AUC={results['roc_auc']}  acc={results['accuracy']}")
    return results


def train_xgb(X_tr, y_tr, X_te, y_te, max_depth=6):
    spw = round((y_tr == 0).sum() / max((y_tr == 1).sum(), 1), 4)
    m = XGBClassifier(
        n_estimators          = 2000,
        max_depth             = max_depth,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        scale_pos_weight      = spw,
        eval_metric           = "logloss",
        early_stopping_rounds = 30,
        random_state          = 42,
        n_jobs                = -1,
    )
    m.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
    return m, spw

## Experimental Design & Architecture

- Architecture A (Segmented Strategy): Hard-split the client base into three groups (flat, moderate, strong) based on their wd_we_l2 distance and train a dedicated, specialized XGBoost model for each segment.

- Architecture B (Global Segment-Aware Strategy): Train a single, unified XGBoost model on the entire dataset, but pass wd_we_l2 and segment dummies directly into the feature matrix so the model implicitly learns the behavioral shifts.

### Architecture A: segment strategy

In [17]:
print("\n" + "═" * 60)
print("A. THREE SEGMENT MODELS")
print("═" * 60)

seg_depth = {"flat": 4, "moderate": 6, "strong": 8}
seg_models, seg_eval = {}, {}

for seg in ["flat", "moderate", "strong"]:
    clients = client_stats[client_stats["segment"] == seg].index.tolist()
    tr = train_all[train_all["client"].isin(clients)]
    te = test_all[test_all["client"].isin(clients)]

    m, spw = train_xgb(
        tr[base_feature_cols], tr["label"],
        te[base_feature_cols], te["label"],
        max_depth=seg_depth[seg],
    )
    res = evaluate(m, te[base_feature_cols], te["label"],
                   label=f"{seg.upper()} ({len(clients)} clients, "
                         f"{len(tr):,} train, {len(te):,} test, "
                         f"spw={spw}, best_iter={m.best_iteration})")
    seg_models[seg] = m
    seg_eval[seg]   = res | {"clients": clients, "te_index": te.index}

    print(classification_report(te["label"], res["y_pred"],
                                 target_names=["Weekday","Weekend"],
                                 zero_division=0))

all_test_idx  = np.concatenate([seg_eval[s]["te_index"] for s in ["flat","moderate","strong"]])
all_y_true    = test_all.loc[all_test_idx, "label"]
all_y_pred_A  = np.concatenate([seg_eval[s]["y_pred"] for s in ["flat","moderate","strong"]])
all_prob_A    = np.concatenate([seg_eval[s]["prob"]   for s in ["flat","moderate","strong"]])

print("Architecture A: combined test scores")
print(f"  Macro-F1   : {f1_score(all_y_true, all_y_pred_A, average='macro'):.4f}")
print(f"  Weekend-F1 : {f1_score(all_y_true, all_y_pred_A, pos_label=1):.4f}")
print(f"  ROC-AUC    : {roc_auc_score(all_y_true, all_prob_A):.4f}")
print(f"  Accuracy   : {(all_y_pred_A == all_y_true.values).mean():.4f}")


════════════════════════════════════════════════════════════
A. THREE SEGMENT MODELS
════════════════════════════════════════════════════════════

  FLAT (122 clients, 117,038 train, 22,431 test, spw=2.4945, best_iter=1999)
  t=0.55  macro-F1=0.6571  weekend-F1=0.5138  AUC=0.7355  acc=0.7171
              precision    recall  f1-score   support

     Weekday       0.81      0.79      0.80     16091
     Weekend       0.50      0.53      0.51      6340

    accuracy                           0.72     22431
   macro avg       0.65      0.66      0.66     22431
weighted avg       0.72      0.72      0.72     22431


  MODERATE (126 clients, 131,009 train, 23,184 test, spw=2.4951, best_iter=1999)
  t=0.59  macro-F1=0.7507  weekend-F1=0.6366  AUC=0.8501  acc=0.8029
              precision    recall  f1-score   support

     Weekday       0.85      0.88      0.86     16632
     Weekend       0.66      0.61      0.64      6552

    accuracy                           0.80     23184
   macro a

### Architecture B: global segment-aware model

In [18]:
print("\n" + "═" * 60)
print("B. GLOBAL MODEL + wd_we_l2 + SEGMENT LABEL")
print("═" * 60)

df_global = df.copy()
df_global = df_global.join(
    client_stats[["wd_we_l2", "segment"]], on="client"
)

seg_dummies = pd.get_dummies(df_global["segment"], prefix="seg", dtype=float)
df_global   = pd.concat([df_global, seg_dummies], axis=1)
seg_dummy_cols = seg_dummies.columns.tolist()

global_feature_cols = base_feature_cols + ["wd_we_l2"] + seg_dummy_cols
print(f"Global feature count: {len(global_feature_cols)}")

train_g = df_global[df_global["date"] <  SPLIT_DATE].copy()
test_g  = df_global[df_global["date"] >= SPLIT_DATE].copy()

m_global, spw_g = train_xgb(
    train_g[global_feature_cols], train_g["label"],
    test_g[global_feature_cols],  test_g["label"],
    max_depth=6,
)
print(f"  spw={spw_g}  best_iter={m_global.best_iteration}")

res_global = evaluate(m_global, test_g[global_feature_cols], test_g["label"],
                      label="GLOBAL combined")

print(classification_report(test_g["label"], res_global["y_pred"],
                             target_names=["Weekday","Weekend"],
                             zero_division=0))

print("Architecture B: per-segment breakdown")
for seg in ["flat", "moderate", "strong"]:
    clients = client_stats[client_stats["segment"] == seg].index
    mask = test_g["client"].isin(clients)
    y_te  = test_g.loc[mask, "label"]
    prob  = m_global.predict_proba(test_g.loc[mask, global_feature_cols])[:, 1]

    best_f1, best_t = 0, 0.5
    for t in np.arange(0.30, 0.80, 0.01):
        f1 = f1_score(y_te, (prob >= t).astype(int), average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, round(t, 2)

    y_pred = (prob >= best_t).astype(int)
    print(f"  {seg:8s}: macro-F1={best_f1:.4f}  "
          f"weekend-F1={f1_score(y_te, y_pred, pos_label=1, zero_division=0):.4f}  "
          f"AUC={roc_auc_score(y_te, prob):.4f}  "
          f"t={best_t}  n={mask.sum():,}")


════════════════════════════════════════════════════════════
B. GLOBAL MODEL + wd_we_l2 + SEGMENT LABEL
════════════════════════════════════════════════════════════
Global feature count: 64
  spw=2.4948  best_iter=1996

  GLOBAL combined
  t=0.62  macro-F1=0.7458  weekend-F1=0.6215  AUC=0.848  acc=0.8066
              precision    recall  f1-score   support

     Weekday       0.84      0.90      0.87     48827
     Weekend       0.70      0.56      0.62     19236

    accuracy                           0.81     68063
   macro avg       0.77      0.73      0.75     68063
weighted avg       0.80      0.81      0.80     68063

Architecture B: per-segment breakdown
  flat    : macro-F1=0.6588  weekend-F1=0.5191  AUC=0.7430  t=0.54  n=22,431
  moderate: macro-F1=0.7488  weekend-F1=0.6281  AUC=0.8464  t=0.62  n=23,184
  strong  : macro-F1=0.8295  weekend-F1=0.7483  AUC=0.9237  t=0.63  n=22,448


## Final Summary

While the overall difference in evaluation performance between the two approaches is slight, the single combined segment-aware model (Architecture B) consistently outperforms the three separate models across all segments, achieving final accuracy of ~80.6%, compared to 79.7%. Based on these results, this global segment-aware modelling approach will be chosen as the final model.

A substantial portion of the client population falls into the "flat" behavioral segment, where the wd_we_l2 metric is near zero. This presents a major challange. For these consumers, the statistical distribution of hourly electricity load on weekends is nearly identical on weekdays, indicating continuous consumption (e.g., industrial plants or always-on infrastructure). The segments with more variable weekly energy consumption perform much better. This suggest that XGBoost may not be the most optimal approach for this problem, pointing towards architectures that are able to detect more subtle shift, like LSTM or utilizing Transformers.

In [19]:
print("\n" + "═" * 60)
print("FINAL COMPARISON")
print("═" * 60)

rows = []
for seg in ["flat", "moderate", "strong"]:
    r = seg_eval[seg]
    rows.append({
        "arch"       : f"A — segmented ({seg})",
        "n_clients"  : len(seg_eval[seg]["clients"]),
        "macro_f1"   : r["macro_f1"],
        "weekend_f1" : r["weekend_f1"],
        "roc_auc"    : r["roc_auc"],
        "accuracy"   : r["accuracy"],
    })

rows.append({
    "arch"       : "A: segmented (combined)",
    "n_clients"  : 370,
    "macro_f1"   : round(f1_score(all_y_true, all_y_pred_A, average="macro"), 4),
    "weekend_f1" : round(f1_score(all_y_true, all_y_pred_A, pos_label=1), 4),
    "roc_auc"    : round(roc_auc_score(all_y_true, all_prob_A), 4),
    "accuracy"   : round((all_y_pred_A == all_y_true.values).mean(), 4),
})

rows.append({
    "arch"       : "B: global + L2 + segment",
    "n_clients"  : 370,
    "macro_f1"   : res_global["macro_f1"],
    "weekend_f1" : res_global["weekend_f1"],
    "roc_auc"    : res_global["roc_auc"],
    "accuracy"   : res_global["accuracy"],
})

summary = pd.DataFrame(rows).set_index("arch")
print(summary.to_string())


════════════════════════════════════════════════════════════
FINAL COMPARISON
════════════════════════════════════════════════════════════
                          n_clients  macro_f1  weekend_f1  roc_auc  accuracy
arch                                                                        
A — segmented (flat)            122    0.6571      0.5138   0.7355    0.7171
A — segmented (moderate)        126    0.7507      0.6366   0.8501    0.8029
A — segmented (strong)          122    0.8386      0.7659   0.9307    0.8713
A: segmented (combined)         370    0.7479      0.6364   0.8497    0.7972
B: global + L2 + segment        370    0.7458      0.6215   0.8480    0.8066


## Baseline comparision
To check the validity of feature engineering a comparision between the achriturectures and a baseline architecture trained only on the hourly data was done, showing a considerable improvement.

In [20]:
m_baseline, spw_base = train_xgb(
    train_all[hour_cols], train_all["label"],
    test_all[hour_cols],  test_all["label"],
    max_depth=6, 
)
res_baseline = evaluate(m_baseline, test_all[hour_cols], test_all["label"])

In [27]:
total_clients = df["client"].nunique()
rows = []

rows.append({
    "arch"       : "Baseline — Raw Hours Only",
    "n_clients"  : total_clients,
    "macro_f1"   : res_baseline["macro_f1"],
    "weekend_f1" : res_baseline["weekend_f1"],
    "roc_auc"    : res_baseline["roc_auc"],
    "accuracy"   : res_baseline["accuracy"]
})

for seg in ["flat", "moderate", "strong"]:
    r = seg_eval[seg]
    rows.append({
        "arch"       : f"Arch A — Segmented ({seg.capitalize()})",
        "n_clients"  : len(r["clients"]),
        "macro_f1"   : r["macro_f1"],
        "weekend_f1" : r["weekend_f1"],
        "roc_auc"    : r["roc_auc"],
        "accuracy"   : r["accuracy"],
    })
    
rows.append({
    "arch"       : "Arch A — Segmented (Combined)",
    "n_clients"  : total_clients,
    "macro_f1"   : round(f1_score(all_y_true, all_y_pred_A, average="macro"), 4),
    "weekend_f1" : round(f1_score(all_y_true, all_y_pred_A, pos_label=1), 4),
    "roc_auc"    : round(roc_auc_score(all_y_true, all_prob_A), 4),
    "accuracy"   : round((all_y_pred_A == all_y_true.values).mean(), 4),
})

rows.append({
    "arch"       : "Arch B:  Global + L2 + Segment",
    "n_clients"  : total_clients,
    "macro_f1"   : res_global["macro_f1"],
    "weekend_f1" : res_global["weekend_f1"],
    "roc_auc"    : res_global["roc_auc"],
    "accuracy"   : res_global["accuracy"],
})


summary_df = pd.DataFrame(rows).set_index("arch")
print(summary_df.to_string())

                                n_clients  macro_f1  weekend_f1  roc_auc  accuracy
arch                                                                              
Baseline — Raw Hours Only             370    0.7075      0.5844   0.8055    0.7594
Arch A — Segmented (Flat)             122    0.6571      0.5138   0.7355    0.7171
Arch A — Segmented (Moderate)         126    0.7507      0.6366   0.8501    0.8029
Arch A — Segmented (Strong)           122    0.8386      0.7659   0.9307    0.8713
Arch A — Segmented (Combined)         370    0.7479      0.6364   0.8497    0.7972
Arch B:  Global + L2 + Segment        370    0.7458      0.6215   0.8480    0.8066
